# অলীকবচন — Bengali LLM Hallucination Detection: Inference Notebook

**Team:** Outliers  |  **Phase-2-ready end-to-end inference** (raw `test.csv` → `submission.csv`)

## Approach (documentation — required by the rulebook)
A **router + open-weight LLM-judge** pipeline. No API calls, all inference local.

1. **Routing.** Each test item is routed to one of three branches:
   `grounded` (context present), `closedbook` (context `[NULL]`/empty), `math` (numeric-reasoning keywords).
2. **Grounded judge.** Qwen2.5-14B-Instruct (AWQ 4-bit) is prompted (Bengali, few-shot) to check that the
   response (a) answers the *type* of question asked, (b) is supported by the context, and (c) does not
   contradict world knowledge — verbatim context overlap alone is treated as insufficient (a known trap).
3. **Closed-book judge + retrieval.** BM25 retrieval (pure-python, offline) over a Bengali Wikipedia
   passage corpus (attached as a public Kaggle dataset — see External Data declaration below); retrieved
   evidence is injected into the judge prompt. Falls back gracefully to knowledge-only judging if the
   corpus is not attached.
4. **Math branch.** The model *solves the problem step-by-step itself*, then compares its answer to the
   candidate answer (solve-and-compare — never "does this look right?").
5. **Self-consistency.** Each verdict is sampled `VOTES` times (T=0.7); the score is the vote fraction.
6. **Decision thresholds** per route are calibrated on the released labeled sample split
   (max F1 on the hallucinated class); falls back to safe defaults biased toward predicting 0 when unsure.

## Rule compliance
- Open-weight model only, loaded from an attached Kaggle dataset/model — **no external APIs, no internet**.
- Single P100 / T4×2, runtime well under 9 h (see runtime guard cell); model weights ≪ 50 GB on disk.
- No hardcoded row counts, row order, or IDs — schema-robust loading (re-run policy compliant).
- External data: Bengali Wikipedia (public, CC BY-SA; dump `wikimedia/wikipedia 20231101.bn`),
  declared and published per the External Data Policy.
- The test set is never used for training/fine-tuning; the labeled sample split is used only to
  calibrate decision thresholds at inference time.

## Required attachments
1. The competition dataset (provides `test.csv`, and `train.csv` if released there).
2. `Qwen/Qwen2.5-14B-Instruct-AWQ` — via Kaggle Models, or a dataset containing the HF snapshot.
3. *(optional but recommended, ≈ +4-6 F1)* Bengali Wikipedia passages parquet built by
   `prep_build_wiki_index.ipynb` (public dataset).
4. *(optional, speed)* a dataset with vLLM wheels for offline `pip install --no-index`; without it the
   notebook falls back to `transformers` + bitsandbytes 4-bit (slower → votes auto-reduced).


In [ ]:
# ------- engine install: MUST run before torch is imported anywhere ----------
# vllm compiled against a newer torch cannot be imported into a kernel where the
# old torch is already loaded. Installing here (cell 0) or in the Dependency
# Manager keeps the install ahead of any torch import. Offline (Phase 2): both
# installs are skipped; attached wheels / the fallback engine take over.
import importlib.util, subprocess, glob as _g

if importlib.util.find_spec("vllm") is None:
    whls = sorted(_g.glob("/kaggle/input/**/vllm*.whl", recursive=True))
    if whls:
        subprocess.run(["pip", "install", "-q", "--no-index",
                        "--find-links", whls[0].rsplit("/", 1)[0], "vllm"], check=False)
    else:
        print("installing vllm (internet ON needed; a failure here is OK -> fallback engine)")
        subprocess.run(["pip", "install", "-q", "vllm"], check=False)

if importlib.util.find_spec("bitsandbytes") is None:  # fallback engine deps (small, torch untouched)
    subprocess.run(["pip", "install", "-q", "-U", "bitsandbytes", "accelerate"], check=False)
print("engine install cell done")


In [ ]:
import os, re, glob, json, gc, time
import numpy as np, pandas as pd

# ------------------------------- configuration -------------------------------
VOTES            = 3          # self-consistency samples per item (auto-reduced on slow fallback engine)
MAX_CTX_CHARS    = 2800       # truncate very long contexts
EVIDENCE_K       = 4          # retrieved wiki passages per closed-book item
EVIDENCE_CHARS   = 500        # chars per passage
BATCH_FALLBACK   = 8          # batch size for the transformers fallback engine
CALIBRATE        = True       # calibrate thresholds on the labeled sample split if it is attached
# Defaults biased toward predicting 0 (hallucinated) when unsure — the metric is F1 on class 0.
THRESHOLDS       = {"grounded": 0.50, "closedbook": 0.60, "math": 0.50}

# Judge preference order: 32B first (bigger judge = higher ceiling), 14B fallback.
JUDGE_CANDIDATES = ["Qwen/Qwen2.5-32B-Instruct-AWQ", "Qwen/Qwen2.5-14B-Instruct-AWQ"]

# Metric to optimize thresholds for. Rulebook says F1 on class 0; Kaggle Evaluation
# tab says macro-F1. Diagnostic: submit all-zeros once — ~0.62 => "f1_0", ~0.31 => "macro".
METRIC           = "f1_0"

def find_test_csv():
    # real layout observed: /kaggle/input/competitions/bengali-hallucination/"test set.csv"
    cands = [p for p in sorted(glob.glob("/kaggle/input/**/*.csv", recursive=True))
             if os.path.basename(p).lower().replace(" ", "").startswith("test")]
    assert cands, "test csv not found under /kaggle/input (looked for test*.csv / 'test set.csv')"
    return cands[0]

def find_labeled_sample():
    """Labeled sample split for threshold calibration: train*.csv or dataset samples.json."""
    for p in sorted(glob.glob("/kaggle/input/**/*", recursive=True)):
        base = os.path.basename(p).lower()
        if base.replace(" ", "").startswith("train") and base.endswith(".csv"):
            return p
    for p in sorted(glob.glob("/kaggle/input/**/*.json", recursive=True)):
        if "sample" in os.path.basename(p).lower():
            return p
    return None

def find_model_dir():
    """Locally attached snapshot (Phase 2 requirement). None -> use JUDGE_CANDIDATES via HF."""
    env = os.environ.get("MODEL_DIR", "")
    if env and os.path.exists(os.path.join(env, "config.json")):
        return env
    for cfg in sorted(glob.glob("/kaggle/input/**/config.json", recursive=True)):
        try:
            c = json.load(open(cfg))
        except Exception:
            continue
        blob = (str(c.get("architectures", "")) + str(c.get("model_type", ""))).lower()
        if "qwen2" in blob:
            return os.path.dirname(cfg)
    print("no local model snapshot -> will pull from Hugging Face (needs internet ON; "
          "attach a snapshot dataset for the Phase 2 offline re-run)")
    return None

def find_wiki_parquet():
    """Retrieval corpus: our prep parquet, or any attached public Bengali
    wiki/GK dataset (e.g. shazol's Bangla Wikipedia) in parquet/csv/json form."""
    hits = sorted(glob.glob("/kaggle/input/**/*passages*.parquet", recursive=True))
    if hits:
        return hits[0]
    cand = []
    for pat in ("*.parquet", "*.csv", "*.json", "*.jsonl"):
        for p in glob.glob(f"/kaggle/input/**/{pat}", recursive=True):
            pl = p.lower()
            if any(k in pl for k in ("wiki", "wikipedia", "bangla-gk", "banglagk"))                     and os.path.getsize(p) > 1_000_000:
                cand.append(p)
    return sorted(cand, key=os.path.getsize)[-1] if cand else None

TEST_CSV  = find_test_csv()
SAMPLE_LB = find_labeled_sample()
MODEL_DIR = find_model_dir()
WIKI_PQ   = find_wiki_parquet()
print("test csv    :", TEST_CSV)
print("labeled samp:", SAMPLE_LB or "NOT FOUND -> default thresholds")
print("model       :", MODEL_DIR)
print("wiki corpus :", WIKI_PQ or "NOT ATTACHED -> knowledge-only closed-book judging")


In [ ]:
# ------------------------- schema-robust test loading ------------------------
# Re-run policy: never assume row count, row order, or specific IDs.
test = pd.read_csv(TEST_CSV)
low = {c.lower().strip(): c for c in test.columns}
COL_CTX = low.get("context")
COL_Q   = low.get("prompt_bn") or low.get("prompt")
COL_R   = low.get("response_bn") or low.get("response")
COL_ID  = low.get("id")
assert COL_Q and COL_R, f"unexpected schema: {list(test.columns)}"
ids = test[COL_ID].values if COL_ID else np.arange(1, len(test) + 1)
print(f"{len(test)} test rows; columns: {list(test.columns)}")


In [ ]:
# ---------------------------------- routing ----------------------------------
MATH_KW = re.compile(
    r"শতকরা|সম্ভাবনা|গুণিতক|মৌলিক|ধারাটি|সমষ্টি|যোগফল|বর্গ(?:মূল|ইঞ্চি|ফুট)?|"
    r"ক্ষতি হ|লাভ হ|বয়স|কত ?দিনে|গড়|অনুপাত|ভগ্নাংশ|সুদ|আসল|"
    r"[0-9০-৯]\s*[+\-*/=]|√|x\s*[*+=]|\*\*|%"
)
BN2EN = str.maketrans("০১২৩৪৫৬৭৮৯", "0123456789")

def has_context(v):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return False
    return str(v).strip() not in ("", "[NULL]", "NULL", "nan")

def route_row(ctx, q):
    if MATH_KW.search(str(q)):
        return "math"
    return "grounded" if has_context(ctx) else "closedbook"

routes = np.array([
    route_row(test[COL_CTX].iloc[i] if COL_CTX else None, test[COL_Q].iloc[i])
    for i in range(len(test))
])
print(pd.Series(routes).value_counts().to_dict())


In [ ]:
# ----------------------- offline BM25 over Bengali wiki ----------------------
TOK_RE = re.compile(r"[\wঀ-৿]+")
STOP = set("কী কি কে কার কাকে কোন কোথায় কবে হয় ছিল ছিলেন এর একটি ও এবং থেকে সালে করে করা হয় হয়েছিল যে এই".split())

def tokenize(s):
    return [t for t in TOK_RE.findall(str(s).translate(BN2EN).lower()) if t not in STOP and len(t) > 1]

class BM25:
    def __init__(self, docs, k1=1.5, b=0.75):
        self.docs, self.k1, self.b = docs, k1, b
        self.post, self.dl = {}, np.zeros(len(docs), dtype=np.float32)
        for i, d in enumerate(docs):
            toks = tokenize(d)
            self.dl[i] = len(toks) or 1
            tf = {}
            for t in toks:
                tf[t] = tf.get(t, 0) + 1
            for t, f in tf.items():
                self.post.setdefault(t, []).append((i, f))
        self.avgdl = float(self.dl.mean()) if len(docs) else 1.0
        self.N = len(docs)

    def top(self, query, k=4):
        scores = {}
        for t in set(tokenize(query)):
            plist = self.post.get(t)
            if not plist:
                continue
            idf = np.log(1 + (self.N - len(plist) + 0.5) / (len(plist) + 0.5))
            for i, f in plist:
                denom = f + self.k1 * (1 - self.b + self.b * self.dl[i] / self.avgdl)
                scores[i] = scores.get(i, 0.0) + idf * f * (self.k1 + 1) / denom
        best = sorted(scores.items(), key=lambda x: -x[1])[:k]
        return [self.docs[i] for i, _ in best]

def load_corpus(path):
    """Load any reasonable corpus file into a list of <=900-char passages."""
    if path.endswith(".parquet"):
        df_ = pd.read_parquet(path)
    elif path.endswith(".csv"):
        df_ = pd.read_csv(path, on_bad_lines="skip")
    else:
        try:
            df_ = pd.read_json(path, lines=path.endswith(".jsonl"))
        except ValueError:
            df_ = pd.read_json(path, lines=True)
    # choose the longest-average string column as the text column
    obj = [c for c in df_.columns if df_[c].dtype == object]
    txt_col = max(obj, key=lambda c: df_[c].astype(str).str.len().mean()) if obj else df_.columns[-1]
    title_col = next((c for c in obj if c != txt_col and "title" in c.lower()), None)
    out = []
    for _, row in df_.iterrows():
        text = str(row[txt_col])
        title = (str(row[title_col]) + ": ") if title_col else ""
        for chunk_start in range(0, min(len(text), 1800), 900):   # first 2 chunks per article
            chunk = text[chunk_start:chunk_start + 900].strip()
            if len(chunk) >= 80:
                out.append(title + chunk)
        if len(out) >= 800_000:
            break
    return out

bm25 = None
if WIKI_PQ:
    t0 = time.time()
    passages = load_corpus(WIKI_PQ)
    bm25 = BM25(passages)
    gc.collect()
    print(f"BM25 index over {len(passages)} passages in {time.time()-t0:.0f}s")


In [ ]:
# ------------------------------- judge engine --------------------------------
import torch
N_GPU = max(torch.cuda.device_count(), 1)
from transformers import AutoTokenizer

ENGINE_KIND, LLM_OBJ, MODEL_SRC, tok = None, None, None, None
SOURCES = ([MODEL_DIR] if MODEL_DIR else []) + JUDGE_CANDIDATES

def _try_vllm():
    global ENGINE_KIND, LLM_OBJ, MODEL_SRC
    try:
        import vllm  # noqa  (installed by cell 0 / Dependency Manager, BEFORE torch import)
    except ImportError:
        print("vllm unavailable -> transformers fallback")
        return False
    from vllm import LLM
    for src in SOURCES:
        try:
            quant = "awq" if "awq" in src.lower() else None
            LLM_OBJ = LLM(model=src, quantization=quant, dtype="half",
                          tensor_parallel_size=N_GPU, max_model_len=8192,
                          gpu_memory_utilization=0.92, enforce_eager=True)
            ENGINE_KIND, MODEL_SRC = "vllm", src
            return True
        except Exception as e:
            print(f"vllm failed for {src}: {type(e).__name__} {str(e)[:150]}")
    return False

def _load_fallback():
    """transformers engine. AWQ checkpoints need awq kernels which Kaggle lacks,
    so on failure we switch to the fp16 repo + bitsandbytes 4-bit."""
    global ENGINE_KIND, LLM_OBJ, VOTES, MODEL_SRC
    from transformers import AutoModelForCausalLM, BitsAndBytesConfig
    src = MODEL_DIR or "Qwen/Qwen2.5-14B-Instruct-AWQ"
    try:
        assert "awq" in src.lower(), "not an AWQ checkpoint"
        LLM_OBJ = AutoModelForCausalLM.from_pretrained(
            src, device_map="auto", dtype=torch.float16, low_cpu_mem_usage=True)
        MODEL_SRC = src
    except Exception as e:
        print(f"AWQ via transformers failed ({type(e).__name__}) -> fp16 repo + bnb 4-bit")
        os.system("pip install -q -U bitsandbytes accelerate")  # small; does not touch torch
        MODEL_SRC = "Qwen/Qwen2.5-14B-Instruct"
        bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
                                 bnb_4bit_quant_type="nf4")
        LLM_OBJ = AutoModelForCausalLM.from_pretrained(
            MODEL_SRC, device_map="auto", quantization_config=bnb, low_cpu_mem_usage=True)
    LLM_OBJ.eval()
    ENGINE_KIND = "hf"
    VOTES = 1  # fallback engine is slow -> greedy single vote keeps us far under 9h
    print("transformers fallback engine; VOTES ->", VOTES)

if not _try_vllm():
    _load_fallback()

tok = AutoTokenizer.from_pretrained(MODEL_SRC)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
tok.padding_side = "left"

def chatfmt(user_msg):
    msgs = [
        {"role": "system", "content": "তুমি বাংলা ভাষার একজন নির্ভুল ও কঠোর ফ্যাক্ট-চেকার। তোমার কাজ: একটি প্রশ্নের প্রার্থী উত্তর বিশ্বস্ত (1) নাকি হ্যালুসিনেটেড/ভুল (0) তা নির্ধারণ করা।"},
        {"role": "user", "content": user_msg},
    ]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

print("engine:", ENGINE_KIND, "| model:", MODEL_SRC)

def generate(prompts, max_tokens, votes):
    """Return list[list[str]] — `votes` completions per prompt."""
    if ENGINE_KIND == "vllm":
        from vllm import SamplingParams
        sp = (SamplingParams(n=votes, temperature=0.7, top_p=0.9, max_tokens=max_tokens)
              if votes > 1 else SamplingParams(temperature=0.0, max_tokens=max_tokens))
        outs = LLM_OBJ.generate(prompts, sp)
        return [[o.text for o in out.outputs] for out in outs]
    results = []
    for s in range(0, len(prompts), BATCH_FALLBACK):
        chunk = prompts[s:s + BATCH_FALLBACK]
        enc = tok(chunk, return_tensors="pt", padding=True, truncation=True,
                  max_length=7168).to(LLM_OBJ.device)
        with torch.no_grad():
            gen = LLM_OBJ.generate(
                **enc, max_new_tokens=max_tokens,
                do_sample=votes > 1, temperature=0.7 if votes > 1 else None,
                top_p=0.9 if votes > 1 else None, num_return_sequences=votes,
                pad_token_id=tok.eos_token_id)
        gen = gen[:, enc["input_ids"].shape[1]:]
        texts = tok.batch_decode(gen, skip_special_tokens=True)
        for i in range(len(chunk)):
            results.append(texts[i * votes:(i + 1) * votes])
        if s % (BATCH_FALLBACK * 20) == 0:
            print(f"  fallback progress {s}/{len(prompts)}", flush=True)
    return results


In [ ]:
# --------------------------- prompt templates (bn) ---------------------------
GROUNDED_T = (
    "নিয়ম:\n"
    "- উত্তরটি বিশ্বস্ত (1) হবে কেবল যদি এটি প্রশ্নে ঠিক যা চাওয়া হয়েছে সেটিই দেয় এবং প্রসঙ্গ ও বাস্তব তথ্যের সাথে মেলে।\n"
    "- প্রসঙ্গ থেকে হুবহু শব্দ কপি করা থাকলেও প্রশ্নের সাথে অসঙ্গত হলে সেটি হ্যালুসিনেটেড (0)। সাল চাইলে সাল দিতে হবে, নাম চাইলে নাম।\n"
    "- প্রশ্নের ভেতরের অনুমান প্রসঙ্গ সমর্থন না করলে সতর্ক হও।\n\n"
    "উদাহরণ ১ — প্রসঙ্গ: \"...গ্রেট বালটোরা পৃথিবীর দীর্ঘতম হিমবাহগুলোর একটি। এর দৈর্ঘ্য প্রায় ৫৮ কিলোমিটার।\" "
    "প্রশ্ন: \"পৃথিবীর দীর্ঘতম হিমবাহটির দৈর্ঘ্য কত?\" উত্তর: \"প্রায় ৫৮ কিলোমিটার দীর্ঘ।\" রায়: 0 "
    "(প্রসঙ্গে বালটোরা অন্যতম দীর্ঘ, দীর্ঘতম নয়)\n"
    "উদাহরণ ২ — প্রশ্ন: \"তারেক মাসুদ পরিচালিত সর্বশেষ চলচ্চিত্রটি কত সালে মুক্তি পায়?\" উত্তর: \"রানওয়ে\" রায়: 0 "
    "(সাল চাওয়া হয়েছে, নাম দেওয়া হয়েছে)\n"
    "উদাহরণ ৩ — প্রসঙ্গ: \"...মেহদী হাসান খান ২০০৩ সালে অভ্র কীবোর্ড তৈরির কাজ শুরু করেন...\" "
    "প্রশ্ন: \"অভ্র কিবোর্ড কে উদ্ভাবন করেন?\" উত্তর: \"মেহদী হাসান খান\" রায়: 1\n"
    "উদাহরণ ৪ — প্রসঙ্গ: \"...এই অঞ্চল ৭০০-এর অধিক দ্বীপপুঞ্জ...দ্বারা গঠিত।\" "
    "প্রশ্ন: \"ক্যারেবীয় মোট কয়টি দ্বীপপুঞ্জের সমন্বয়ে গঠিত?\" উত্তর: \"৭০০-এর অধিক\" রায়: 1\n\n"
    "এখন মূল্যায়ন করো। প্রথমে সর্বোচ্চ দুই বাক্যে তোমার যুক্তি লেখো, তারপর শেষ লাইনে ঠিক এই ফরম্যাটে লেখো — "
    "রায়: 1 (বিশ্বস্ত) অথবা রায়: 0 (হ্যালুসিনেটেড)।\n\n"
    "প্রসঙ্গ: {ctx}\n\nপ্রশ্ন: {q}\nউত্তর: {r}"
)

CLOSEDBOOK_T = (
    "প্রার্থী উত্তরটি তোমার জ্ঞান{ev_clause} অনুযায়ী সঠিক কিনা যাচাই করো।\n"
    "নিয়ম: ভুল ব্যক্তি/সাল/স্থান/সংখ্যা, আংশিক ভুল, বা বানোয়াট হলে 0। সঠিক ও প্রশ্নের সাথে সঙ্গত হলে 1।\n\n"
    "উদাহরণ — প্রশ্ন: \"আয়তনে পৃথিবীর সবচেয়ে ছোট দেশ?\" উত্তর: \"মালদ্বীপ\" রায়: 0 (সঠিক উত্তর ভ্যাটিকান সিটি)\n"
    "উদাহরণ — প্রশ্ন: \"এডেন কোন দেশের সমুদ্রবন্দর?\" উত্তর: \"ইয়েমেন\" রায়: 1\n"
    "উদাহরণ — প্রশ্ন: \"আটলান্টিক সনদে যুক্তরাষ্ট্র ও ব্রিটিশদের পক্ষে স্বাক্ষর করেন কে কে?\" "
    "উত্তর: \"জিমি কার্টার ও রাণী দ্বিতীয় এলিজাবেথ\" রায়: 0 (রুজভেল্ট ও চার্চিল)\n"
    "উদাহরণ — প্রশ্ন: \"'আগুনপাখি' উপন্যাসের রচয়িতা কে?\" উত্তর: \"হাসান আজিজুল হক\" রায়: 1\n"
    "{ev_block}"
    "প্রথমে সর্বোচ্চ দুই বাক্যে যুক্তি লেখো (সঠিক উত্তরটি কী তা মনে করার চেষ্টা করো), তারপর শেষ লাইনে ঠিক এই "
    "ফরম্যাটে লেখো — রায়: 1 অথবা রায়: 0।\n\nপ্রশ্ন: {q}\nউত্তর: {r}"
)

MATH_T = (
    "নিচের গাণিতিক প্রশ্নটি ধাপে ধাপে নিজে সমাধান করো। তারপর তোমার পাওয়া উত্তরের সাথে প্রার্থী উত্তরটি মিলিয়ে দেখো "
    "(একই মান ভিন্নভাবে লেখা থাকলে সেটি মিল ধরো, যেমন ১/২ = 0.5)।\n"
    "সবশেষে ঠিক এই ফরম্যাটে একটি লাইন লেখো — রায়: 1 (মিললে) অথবা রায়: 0 (না মিললে)।\n\n"
    "উদাহরণ — প্রশ্ন: \"এক ব্যক্তি তার স্ত্রীর চেয়ে ৫ বছরের বড়। স্ত্রীর বয়স ছেলের বয়সের ৪ গুণ। ৫ বছর পরে ছেলের বয়স "
    "১২ হলে ব্যক্তির বর্তমান বয়স কত?\" প্রার্থী উত্তর: \"৬৫ বছর\"\n"
    "সমাধান: ৫ বছর পরে ছেলের বয়স ১২ ⇒ এখন ৭। স্ত্রী = ৪×৭ = ২৮। ব্যক্তি = ২৮+৫ = ৩৩। প্রার্থী বলেছে ৬৫ ≠ ৩৩।\n"
    "রায়: 0\n\n"
    "প্রশ্ন: {q}\nপ্রার্থী উত্তর: {r}\nসমাধান:"
)

# Bengali tokenizes expensively (~1 token/char in Qwen BPE) -> exact token-level guard.
MAX_INPUT_TOKENS = 7000   # max_model_len 8192 minus output budget and margin

def _ntokens(p):
    try:
        return len(tok(p)["input_ids"])
    except Exception:       # tokenizer unavailable (e.g. dry run) -> pessimistic estimate
        return len(p)

def build_prompt(df, cctx, cq, cr, rts, i):
    q = str(df[cq].iloc[i])[:1500]
    r = str(df[cr].iloc[i])[:800]
    rt = rts[i]
    if rt == "math":
        return chatfmt(MATH_T.format(q=q, r=r))
    ctx_chars, ev_chars = MAX_CTX_CHARS, EVIDENCE_CHARS
    hits = bm25.top(q + " " + r, k=EVIDENCE_K) if (rt == "closedbook" and bm25 is not None) else []
    while True:
        if rt == "grounded":
            ctx = str(df[cctx].iloc[i])[:ctx_chars]
            p = chatfmt(GROUNDED_T.format(ctx=ctx, q=q, r=r))
        else:
            ev_clause, ev_block = "", ""
            if hits:
                ev_clause = " ও নিচের তথ্যসূত্র"
                ev_block = "তথ্যসূত্র:\n" + "\n".join(
                    f"- {h[:ev_chars]}" for h in hits) + "\n\n"
            p = chatfmt(CLOSEDBOOK_T.format(ev_clause=ev_clause, ev_block=ev_block, q=q, r=r))
        if _ntokens(p) <= MAX_INPUT_TOKENS or ctx_chars <= 150:
            return p
        ctx_chars //= 2
        ev_chars = max(ev_chars // 2, 100)

VERDICT_RE = re.compile(r"রায়\s*[:ঃ]?\s*([01])")

def parse_verdict(text, rt):
    """All routes reason first, then end with 'রায়: X' -> take the LAST verdict.
    Returns None when no verdict is found (e.g. truncated output) — the caller
    treats None as an abstention instead of silently counting it as 0."""
    t = str(text).translate(BN2EN)
    m = VERDICT_RE.findall(t)
    if m:
        return int(m[-1])
    m = re.findall(r"\b[01]\b", t)
    return int(m[-1]) if m else None


In [ ]:
# ---------------------------- scoring the test set ---------------------------
t_start = time.time()

def score_frame(df, cctx, cq, cr, rts, votes):
    """Score every row of df; returns np.array of faithfulness scores in [0,1]."""
    n = len(df)
    scores = np.zeros(n, dtype=np.float32)
    short = [i for i in range(n) if rts[i] != "math"]
    long_ = [i for i in range(n) if rts[i] == "math"]
    # 352 tokens for the reasoning routes: Bengali costs ~1 token/char, and truncating
    # the output BEFORE the model writes 'রায়: X' poisons the votes (learned the hard way).
    n_abstain = 0
    for group, mt in ((short, 352), (long_, 640)):
        if not group:
            continue
        prompts = [build_prompt(df, cctx, cq, cr, rts, i) for i in group]
        outs = generate(prompts, max_tokens=mt, votes=votes)
        for i, comps in zip(group, outs):
            vs = [v for v in (parse_verdict(c, rts[i]) for c in comps) if v is not None]
            scores[i] = float(np.mean(vs)) if vs else 0.5   # full abstention -> neutral, not 0
            n_abstain += (not vs)
    if n_abstain:
        print(f"WARNING: {n_abstain} items had no parseable verdict (score set to 0.5)")
    return scores

# runtime guard: keep a wide margin inside the 9h kernel limit
if len(test) * VOTES > 40_000:
    VOTES = 1
    print("large test set -> VOTES reduced to 1")

test_scores = score_frame(test, COL_CTX, COL_Q, COL_R, routes, VOTES)
print(f"scored {len(test)} items in {(time.time()-t_start)/60:.1f} min")


In [ ]:
# ------------- threshold calibration on the labeled sample split -------------
def f1_class(y_true, y_pred, cls):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    tp = int(((y_pred == cls) & (y_true == cls)).sum())
    fp = int(((y_pred == cls) & (y_true != cls)).sum())
    fn = int(((y_pred != cls) & (y_true == cls)).sum())
    return 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) else 0.0

def f1_hallucinated(y, p):
    return f1_class(y, p, 0)

def f1_macro(y, p):
    return (f1_class(y, p, 0) + f1_class(y, p, 1)) / 2

def metric_score(y, p):
    return f1_macro(y, p) if METRIC == "macro" else f1_hallucinated(y, p)

def best_threshold(scores, y):
    best_t, best_f1 = 0.5, -1.0
    for t in np.linspace(0.05, 0.95, 91):
        f1 = metric_score(y, (np.asarray(scores) >= t).astype(int))
        if f1 > best_f1:
            best_t, best_f1 = float(t), f1
    return best_t, best_f1

if CALIBRATE and SAMPLE_LB:
    tr = pd.read_json(SAMPLE_LB) if SAMPLE_LB.endswith(".json") else pd.read_csv(SAMPLE_LB)
    tlow = {c.lower().strip(): c for c in tr.columns}
    if "label" in tlow and "prompt_bn" in tlow and "response_bn" in tlow:
        t_ctx, t_q, t_r = tlow.get("context"), tlow["prompt_bn"], tlow["response_bn"]
        tr_routes = np.array([
            route_row(tr[t_ctx].iloc[i] if t_ctx else None, tr[t_q].iloc[i])
            for i in range(len(tr))])
        tr_scores = score_frame(tr, t_ctx, t_q, t_r, tr_routes, max(VOTES, 1))
        y = tr[tlow["label"]].values
        for rt in ("grounded", "closedbook", "math"):
            m = tr_routes == rt
            if m.sum() >= 10:
                t, f1 = best_threshold(tr_scores[m], y[m])
                THRESHOLDS[rt] = t
                print(f"{rt:<10} n={m.sum():>3}  best_t={t:.2f}  sample-split {METRIC}={f1:.3f}")
        pred = np.array([int(tr_scores[i] >= THRESHOLDS[tr_routes[i]]) for i in range(len(tr))])
        print(f"sample-split overall  F1(0)={f1_hallucinated(y, pred):.4f}  "
              f"macro-F1={f1_macro(y, pred):.4f}  (optimized for: {METRIC})")
        # per-item dump for error analysis and cross-model stacking
        pd.DataFrame({"route": tr_routes, "score": tr_scores, "label": y}).to_csv(
            "calib_scores.csv", index=False)
    else:
        print("sample split found but no label column -> default thresholds", THRESHOLDS)
else:
    print("no labeled sample split attached -> using default thresholds", THRESHOLDS)


In [ ]:
# ------------------------------ write submission -----------------------------
labels = np.array([int(test_scores[i] >= THRESHOLDS[routes[i]]) for i in range(len(test))], dtype=int)

sub = pd.DataFrame({"id": ids, "label": labels})
# strict format checks (rulebook: exactly two columns id,label; every label 0/1; no missing rows)
assert list(sub.columns) == ["id", "label"]
assert len(sub) == len(test)
assert sub["label"].isin([0, 1]).all()
assert sub["id"].notna().all()
sub.to_csv("submission.csv", index=False)
# per-item scores for later ensembling/stacking with teammate models
pd.DataFrame({"id": ids, "route": routes, "score": test_scores, "label": labels}).to_csv(
    "test_scores.csv", index=False)
print(sub["label"].value_counts().to_dict())
print(f"total runtime: {(time.time()-t_start)/60:.1f} min")
sub.head()


## Reproduction & notes
- Deterministic apart from self-consistency sampling; set `VOTES = 1` for fully greedy deterministic output.
- The pipeline reproduces Phase-1 predictions end-to-end from the raw `test.csv` (Phase-2 requirement).
- Runtime on T4×2 with vLLM: ≈ 30–60 min for ~2.5k items with 3 votes — comfortably inside the 9 h limit.
  The transformers fallback engine reduces to 1 vote and stays inside the limit as well.
- External data declaration: Bengali Wikipedia dump (`wikimedia/wikipedia`, config `20231101.bn`, CC BY-SA),
  processed by the companion prep notebook into a public Kaggle dataset. No test-set-derived data is used.
